# 14.1 Presolve

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

AMPL's presolve phase attempts to simplify a problem instance after it has been generated
but before it is sent to a solver. It runs automatically when a `solve` command is
given or in response to other commands that generate an instance, as explained in Section
A.18.1. Any simplifications that presolve makes are reversed after a solution is returned,
so that you can view the solution in terms of the original problem. Thus presolve normally
proceeds silently behind the scenes. Its effects are only reported when you change
option show_stats from its default value of 0 to 1:

```ampl
ampl: model steelT.mod; data steelT.dat;
ampl: option show_stats 1;
ampl: solve;
Presolve eliminates 2 constraints and 2 variables.

Adjusted problem:

24 variables, all linear
12 constraints, all linear; 38 nonzeros
1 linear objective; 24 nonzeros.

MINOS 5.5: optimal solution found.


15 iterations, objective 515033
```

You can determine which variables and constraints presolve eliminated by testing, as
explained in [Section 14.2](./14_2_retrieving_results_from_solvers.ipynb), to see which have a status of pre:

```ampl
ampl: print {j in 1.._nvars:
ampl?    _var[j].status = "pre"}: _varname[j];
Inv['bands',0]
Inv['coils',0]
ampl: print {i in 1.._ncons:
ampl?    _con[i].status = "pre"}: _conname[i];
Init_Inv['bands']
Init_Inv['coils']
```

You can then use show and `display` to examine the eliminated components.

In this section we introduce the operations of the presolve phase and the options for
controlling it from AMPL. We then explain what presolve does when it detects that no
feasible solution is possible. We will not try to explain the whole presolve algorithm,
however; one of the references at the end of this chapter contains a complete description.

Activities of the presolve phase
To generate a problem instance, AMPL first assigns each variable whatever bounds are
specified in its `var` declaration, or the special bounds -Infinity and Infinity
when no lower or upper bounds are given. The presolve phase then tries to use these
bounds together with the linear constraints to deduce tighter bounds that are still satisfied
by all of the problem's feasible solutions. Concurrently, presolve tries to use the tighter
bounds to detect variables that can be fixed and constraints that can be dropped.

Presolve initially looks for constraints that have only one variable. Equalities of this
kind `fix` a variable, which may then be dropped from the problem. Inequalities specify a
bound for a variable, which may be folded into the existing bounds. In the example of
steelT.mod ([Figure 4-4](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-4)) shown above, presolve eliminates the two constraints generated
from the declaration

```ampl
subject to Initial {p in PROD}:               Inv[p,0] = inv0[p];
```

along with the two variables fixed by these constraints.

Presolve continues by seeking constraints that can be proved redundant by the current
bounds. The constraints eliminated from dietu.mod ([Figure 5-1](../05/5_3_set_operations.ipynb#fig-5-1)) provide an example:

```ampl
ampl: model dietu.mod; data dietu.dat;
ampl: option show_stats 1;
ampl: solve;
Presolve eliminates 3 constraints.

Adjusted problem:

8 variables, all linear
5 constraints, all linear; 39 nonzeros
1 linear objective; 8 nonzeros.

MINOS 5.5: optimal solution found.

5 iterations, objective 74.27382022

ampl: print {i in 1.._ncons:
ampl?    _con[i].status = "pre"}: _conname[i];
Diet_Min['B1']
Diet_Min['B2']
Diet_Max['A']
```

On further investigation, the constraint Diet_Min['B1'] is seen to be redundant
because it is generated from

```ampl
subject to Diet_Min {i in MINREQ}:
       sum {j in FOOD} amt[i,j] * Buy[j] >= n_min[i];
```

with n_min['B1'] equal to zero in the `data`. Clearly this is satisfied by any combination
of the variables, since they all have nonnegative lower bounds. A less trivial case is
given by Diet_Max['A'], which is generated from

```ampl
subject to Diet_Max {i in MAXREQ}:
       sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];
```

By setting each variable to its upper bound on the left-hand side of this constraint, we get
an upper bound on the total amount of the nutrient that any solution can possibly supply.
In particular, for nutrient A:

```ampl
ampl: display sum {j in FOOD} amt['A',j] * f_max[j];
sum{j in FOOD} amt['A',j]*f_max[j] = 2860
```

Since the `data` file gives n_max['A'] as 20000, this is another constraint that cannot
possibly be violated by the variables.

Following these tests, the first part of presolve is completed. The remainder consists
of a series of passes through the problem, each attempting to deduce still tighter variable
bounds from the current bounds and the linear constraints. We present here only one
example of the outcome, for the problem instance generated from multi.mod and
multi.dat (Figures 4-1 and 4-2):

```ampl
ampl:   model multi.mod;
ampl:   data multi.dat;
ampl:   option show_stats 1;
ampl:   solve;
Presolve eliminates 7 constraints and 3 variables.

Adjusted problem:
60 variables, all linear
44 constraints, all linear; 165 nonzeros
1 linear objective; 60 nonzeros.

MINOS 5.5: optimal solution found.

41 iterations, objective 199500
ampl: print {j in 1.._nvars:
ampl?    _var.status[j] = "pre"}: _varname[j];
Trans['GARY','LAN','plate']
Trans['CLEV','LAN','plate']
Trans['PITT','LAN','plate']
ampl: print {i in 1.._ncons:
ampl?    _con[i].status = "pre"}: _conname[i];
Demand['LAN','plate']
Multi['GARY','LAN']
Multi['GARY','WIN']
Multi['CLEV','LAN']
Multi['CLEV','WIN']
Multi['PITT','LAN']
Multi['PITT','WIN']
```

We can see where some of the simplifications come from by expanding the eliminated
demand constraint:

```ampl
ampl: expand Demand['LAN','plate'];
subject to Demand['LAN','plate']:
       Trans['GARY','LAN','plate'] + Trans['CLEV','LAN','plate'] +
       Trans['PITT','LAN','plate'] = 0;
```

Because demand['LAN','plate'] is zero in the `data`, this constraint forces the sum
of three nonnegative variables to be zero, as a result of which all three must have an
upper limit of zero in any solution. Since they already have a lower limit of zero, they
may be fixed and the constraint may be dropped. The other eliminated constraints all
look like this:

```ampl
ampl: expand Multi['GARY','LAN'];
subject to Multi['GARY','LAN']:
       Trans['GARY','LAN','bands'] + Trans['GARY','LAN','coils'] +
       Trans['GARY','LAN','plate'] <= 625;
```

They can be dropped because the sum of the upper bounds of the variables on the left is
less than 625. These variables were not originally given upper bounds in the problem,
however. Instead, the second part of presolve deduced their bounds. For this simple
problem, it is not hard to see how the deduced bounds arise: the amount of any product
shipped along any one link cannot exceed the demand for that product at the destination
of the link. In the case of the destinations LAN and WIN, the total demand for the three
products is less than the limit of 625 on total shipments from any origin, making the
total-shipment constraints redundant.

Controlling the effects of presolve
For more complex problems, presolve's eliminations of variables and constraints may
not be so easy to explain, but they can represent a substantial fraction of the problem.
The time and memory needed to `solve` a problem instance may be reduced considerably
as a result. In rare cases, presolve can also substantially affect the optimal values of the
variables — when there is more than one optimal solution — or interfere with other preprocessing
routines that are built into your solver software. To turn off presolve entirely,
set option presolve to 0; to turn off the second part only, set it to 1. A higher value for
this option indicates the maximum number of passes made in part two of presolve; the
default is 10.
Following presolve, AMPL saves two sets of lower and upper bounds on the variables:
ones that reflect the tightening of the bounds implied by constraints that presolve eliminated,
and ones that reflect further tightening deduced from constraints that presolve
could not eliminate. The problem has the same solution with either set of bounds, but the
overall solution time may be lower with one or the other, depending on the optimization
method in use and the specifics of the problem.

For continuous variables, normally AMPL passes to solvers the first set of bounds, but
you can instruct it to pass the second set by changing option var_bounds to 2 from its
default value of 1. When active-set methods (like the simplex method) are applied, the
second set tends to give rise to more degenerate variables, and hence more degenerate
iterations that may impede progress toward a solution.

For `integer` variables, AMPL rounds any fractional lower bounds up to the next higher
`integer` and any fractional upper bounds down to the next lower `integer`. Due to inaccuracies
of finite-precision computation, however, a bound may be calculated to have a value
that is just slightly different from an `integer`. A lower bound that should be 7, for example,
might be calculated as 7.00000000001, in which case you would not want the bound
to be rounded up to 8! To deal with this possibility, AMPL subtracts the value of option
presolve_inteps from each lower bound, and adds it to each upper bound, before
rounding. If increasing this setting to the value of option presolve_intepsmax
would make a difference to the rounded bounds of any of the variables, AMPL issues a
warning. The default values of presolve_inteps and presolve_intepsmax are 1.0e-6
and 1.0e-5, respectively.

You can examine the first set of presolve bounds by using the suffixes .lb1 and .ub1,
and the second set by .lb2 and .ub2. The original bounds, which are sent to the
solver only if presolve is turned off, are given as .lb0 and .ub0. The suffixes .lb and .ub
give the bound values currently to be passed to the solver, based on the current values
of options presolve and var_bounds.

**Detecting infeasibility in presolve**

If presolve determines that any variable's lower bound is greater than its upper bound,
then there can be no solution satisfying all the bounds and other constraints, and an error
message is printed. For example, here's what would happen to steel3.mod ([Figure 1-5a](../01/tut_1_5.ipynb#fig-1-5a))
if we changed market["bands"] to 500 when we meant 5000:

```ampl
ampl: model steel3.mod;
ampl: data steel3.dat;
ampl: let market["bands"] := 500;
ampl: solve;
inconsistent bounds for var Make['bands']:
              lower bound = 1000 > upper bound = 500;
              difference = 500
```

This is a simple case, because the upper bound on variable Make["bands"] has clearly
been reduced below the lower bound. Presolve's more sophisticated tests can also find
infeasibilities that are not due to any one variable. As an example, consider the constraint
in this `model`:

```ampl
subject to Time: sum {p in PROD} 1/rate[p]*Make[p] <= avail;
```

If we reduce the value of avail to 13 hours, presolve deduces that this constraint can't
possibly be satisfied:

```ampl
ampl: let market["bands"] := 5000;
ampl: let avail := 13;
ampl: solve;
presolve: constraint Time cannot hold:
       body <= 13 cannot be >= 13.2589; difference = -0.258929
```

The "body" of constraint Time is sum {p in PROD} 1/rate[p]*Make[p], the
part that contains the variables (see [Section 12.5](../12/12_5_displaying_solution_values.ipynb)). Thus, given the value of avail that
we have set, the constraint places an upper bound of 13 on the value of the body expression.
On the other hand, if we set each variable in the body expression equal to its lower
bound, we get a lower bound on the value of the body in any feasible solution:

```ampl
ampl: display sum {p in PROD} 1/rate[p]*Make[p].lb2;
sum{p in PROD} 1/rate[p]*(Make[p].lb2) = 13.2589
```

The statement from presolve that body <= 13 cannot be >= 13.2589 is thus reporting
that the upper bound on the body is in conflict with the lower bound, implying that no
solution can satisfy all of the problem's bounds and constraints.

Presolve reports the difference between its two bounds for constraint Time
as -0.258929 (to six digits). Thus in this case we can guess that 13.258929 is,
approximately, the smallest value of avail that allows for a feasible solution,
which we can verify by experiment:

```ampl
ampl: let avail := 13.258929;
ampl: solve;
MINOS 5.5: optimal solution found.

       0 iterations, objective 61750.00214
```

If we make avail just slightly lower, however, we again get the infeasibility message:

```ampl
ampl: let avail := 13.258928;
ampl: solve;
presolve: constraint Time cannot hold:
              body <= 13.2589 cannot be >= 13.2589;
              difference = -5.71429e-07
Setting $presolve_eps >= 6.86e-07 might help.
```

Although the lower bound here is the same as the upper bound to six digits, it is greater
than the upper bound in full precision, as the negative value of the difference indicates.

Typing `solve` a second time in this situation tells AMPL to override presolve and
send the seemingly inconsistent deduced bounds to the solver:

```ampl
ampl: solve;
MINOS 5.5: optimal solution found.

0 iterations, objective 61749.99714
ampl: option display_precision 10;
ampl: display commit, Make;
:     commit       Make :=
bands   1000   999.9998857
coils    500   500
plate    750   750
;
```

MINOS declares that it has found an optimal solution, though with Make["bands"]
being slightly less than its lower bound commit["bands"]! Here MINOS is applying
an internal tolerance that allows small infeasibilities to be ignored; the AMPL/MINOS documentation
explains how this tolerance works and how it can be changed. Each solver
applies feasibility tolerances in its own way, so it's not surprising that a different solver
gives different results:

```ampl
ampl: option solver cplex;
ampl: option send_statuses 0;
ampl: solve;
CPLEX 8.0.0: Bound infeasibility column 'x1'.

infeasible problem.

1 dual simplex iterations (0 in phase I)
```

Here CPLEX has applied its own presolve routine and has detected the same infeasibility
that AMPL did. (You may see a few additional lines about a "suffix" named dunbdd;
this pertains to a direction of unboundedness that you can retrieve via AMPL's solverdefined
suffix feature described in [Section 14.3](./14_3_sending_suffixes_to_solvers.ipynb).)

Situations like this come about when the implied lower and upper bounds on some
variable or constraint body are equal, at least for all practical purposes. Due to imprecision
in the computations, the lower bound may come out slightly greater than the upper
bound, causing AMPL's presolve to report an infeasible problem. To circumvent this difficulty,
you can `reset` the option presolve_eps from its default value of 0 to some
small positive value. Differences between the lower and upper bound are ignored when
they are less than this value. If increasing the current presolve_eps value to a value
no greater than presolve_epsmax would change presolve's handling of the problem,
then presolve displays a message to this effect, such as

```ampl
Setting $presolve_eps >= 6.86e-07 might help.
```

in the example above. The default value of option presolve_eps is zero and
presolve_epsmax is 1.0e-5.

A related situation occurs when imprecision in the computations causes the implied
lower bound on some variable or constraint body to come out slightly lower than the
implied upper bound. Here no infeasibility is detected, but the presence of bounds that
are nearly equal may make the solver's work much harder than necessary. Thus whenever
the upper bound minus the lower bound on a variable or constraint body is positive
but less than the value of option presolve_fixeps, the variable or constraint body is
fixed at the average of the two bounds. If increasing the value of presolve_fixeps
to at most the value of presolve_fixepsmax would change the results of presolve, a
message to this effect is displayed.

The number of separate messages displayed by presolve is limited to the value of
presolve_warnings, which is 5 by default. Increasing option show_stats to 2
may elicit some additional information about the presolve run, including the number of
passes that made a difference to the results and the values to which presolve_eps and
presolve_inteps would have to be increased or decreased to make a difference.